In [5]:
import pandas as pd
import requests
from io import BytesIO

def fetch_data_from_onedrive():
    url = 'https://1drv.ms/x/c/d25227ef9750cee4/EcAql4kIE7xPv_iBm5Msa5kBFy-ZAzjaf0jNREX_JJI-bw?download=1'
    response = requests.get(url)

    if response.status_code == 200:
        try:
            # Read Excel data with proper formatting
            df = pd.read_excel(
                BytesIO(response.content),
                sheet_name='Data In',
                skiprows=6,
                usecols='A:H',
                names=['TIME', 'Temperature', 'Humidity', 'AQI', 'Pressure', 'Wind Speed', 'Rainfall', 'Wind Dir'],
                parse_dates=['TIME']
            )

            # Clean data
            df = df[df['TIME'].notna() & df['Temperature'].apply(lambda x: isinstance(x, (int, float)))]
            numeric_cols = ['Temperature', 'Humidity', 'Pressure', 'Wind Speed', 'Rainfall']
            df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
            df = df.dropna(subset=numeric_cols).drop(columns=['AQI'])

            return df.sort_values('TIME').reset_index(drop=True)

        except Exception as e:
            print(f"Data processing error: {e}")
            return None
    else:
        print(f"Failed to fetch data: HTTP {response.status_code}")
        return None

In [12]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from datetime import datetime, timedelta
import pytz

def process_onedrive_data(raw_data):
    if raw_data is None or raw_data.empty:
        return None

    # Create 5-minute intervals
    raw_data['group'] = (raw_data.index // 5)

    # Aggregate data
    grouped = raw_data.groupby('group').agg({
        'Temperature': ['min', 'max', 'mean'],
        'Humidity': 'mean',
        'Pressure': 'mean',
        'Rainfall': 'mean'
    })

    # Clean column names
    grouped.columns = ['MinTemp', 'MaxTemp', 'Temp', 'Humidity', 'Pressure', 'Rainfall_avg']

    # Add required columns
    grouped['WindGustDir'] = 'E'
    grouped['WindGustSpeed'] = 0.0
    grouped['RainTomorrow'] = grouped['Rainfall_avg'].apply(lambda x: 'Yes' if x > 0 else 'No')

    return grouped[['MinTemp', 'MaxTemp', 'WindGustDir', 'WindGustSpeed',
                    'Humidity', 'Pressure', 'Temp', 'RainTomorrow']]

def prepare_data(data):
    le = LabelEncoder()

    # Encode categorical features
    data['WindGustDir'] = le.fit_transform(data['WindGustDir'])

    # Separate features and target
    x = data[['MinTemp', 'MaxTemp', 'WindGustDir', 'WindGustSpeed',
              'Humidity', 'Pressure', 'Temp']]  # Removed RainTomorrow
    y = data['RainTomorrow']

    return x, y, le

def train_rain_model(x, y):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(x_train, y_train)

    # Print accuracy
    print(f"Model Accuracy: {model.score(x_test, y_test):.2%}")
    return model

def prepare_regression_data(data, feature):
    x, y = [], []
    for i in range(len(data) - 1):
        x.append(data[feature].iloc[i])
        y.append(data[feature].iloc[i + 1])
    return np.array(x).reshape(-1, 1), np.array(y)

def train_regression_model(x, y):
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(x, y)
    return model

def predict_future(model, current_value):
    predictions = [current_value]
    for _ in range(5):
        next_val = model.predict([[predictions[-1]]])[0]
        predictions.append(next_val)
    return predictions[1:]

def weather_view():
    # Fetch and process data
    raw_data = fetch_data_from_onedrive()
    if raw_data is None:
        print("Failed to fetch data from OneDrive")
        return

    processed_data = process_onedrive_data(raw_data)
    if processed_data is None:
        print("No valid data available for processing")
        return

    # Prepare data
    x, y, le = prepare_data(processed_data)

    # Train model
    rain_model = train_rain_model(x, y)

    # Get latest data point (excluding RainTomorrow)
    current_features = processed_data.iloc[-1][['MinTemp', 'MaxTemp', 'WindGustDir',
                                              'WindGustSpeed', 'Humidity', 'Pressure', 'Temp']]

    # Prepare regression models
    x_temp, y_temp = prepare_regression_data(processed_data, 'Temp')
    x_hum, y_hum = prepare_regression_data(processed_data, 'Humidity')

    temp_model = train_regression_model(x_temp, y_temp)
    hum_model = train_regression_model(x_hum, y_hum)

    # Generate predictions
    rain_prediction = rain_model.predict([current_features])[0]
    future_temp = predict_future(temp_model, current_features['Temp'])
    future_humidity = predict_future(hum_model, current_features['Humidity'])

    # Display results
    timezone = pytz.timezone('Asia/Karachi')
    base_time = datetime.now(timezone).replace(second=0, microsecond=0) + timedelta(hours=1)

    print("\n=== Weather Prediction ===")
    print(f"Current Temperature: {current_features['Temp']:.1f}°C")
    print(f"Rain Prediction: {'Yes' if rain_prediction else 'No'}")
    print(f"Weather Prediction: {current_weather['description']}")
    print("\n=== Temperature Forecast ===")
    for i, temp in enumerate(future_temp):
        print(f"Hour {i+1}: {temp:.1f}°C")

    print("\n=== Humidity Forecast ===")
    for i, hum in enumerate(future_humidity):
        print(f"Hour {i+1}: {hum:.1f}%")

if __name__ == "__main__":
    weather_view()

Model Accuracy: 100.00%

=== Weather Prediction ===
Current Temperature: 30.4°C
Rain Prediction: Yes

=== Temperature Forecast ===
Hour 1: 30.4°C
Hour 2: 30.4°C
Hour 3: 30.4°C
Hour 4: 30.4°C
Hour 5: 30.4°C

=== Humidity Forecast ===
Hour 1: 39.3%
Hour 2: 39.4%
Hour 3: 39.3%
Hour 4: 39.4%
Hour 5: 39.3%


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
